# demo_sample_check — which pseudobulk donors have genotypes?

Notebook form of `scripts/01`–`04`. Every step returns a **DataFrame**, so the result is
explorable rather than only printed. See `../README.md` for the reasoning behind the ID
matching — the short version is that a genotype IID is not a person, and
`clinical_core_out/genome_crosswalk.csv` is what makes the two sides comparable.

Run order: paths → (01) pull → (02) donors → (03) overlap → (04) clinical → chart.

> **Before committing:** *Kernel → Restart & Clear Output*. Cell outputs embed donor IDs
> and phenotypes, and this notebook is the only tracked file here that can carry them —
> `pseudobulk/`, `ad-pd-gwas-demo/` and `out/` are all gitignored.

## Paths

The only machine-specific cell. Everything below is relative to these.

In [ ]:
from pathlib import Path

import pandas as pd

HERE = Path("~/projects/amp-ad-pd-wgs-gwas/demo_sample_check").expanduser()  # edit for your checkout
REPO = HERE.parent

PSEUDOBULK = HERE / "pseudobulk"
OUT = HERE / "out"
OUT.mkdir(exist_ok=True)

CROSSWALK = REPO / "clinical_core_out/genome_crosswalk.csv"
GRAIN = REPO / "clinical_core_out/analysis_grain.csv"

# Add a line to check a new callset — that is the whole of "plan step 6".
CALLSETS = {
    "demo_AMP_PD_EUR": HERE / "ad-pd-gwas-demo/FILTERED.AMP_PD_EUR.psam",
    "demo_AMP_AD_EUR": HERE / "ad-pd-gwas-demo/FILTERED.AMP_AD_EUR.psam",
    "AMP-PD_WB-DWGS": REPO / "data/amp-pd-genomics/WB-DWGS/joint_calls/all_chrs_merged.psam",
    "AMP-PD_BR-DSNWGS": REPO / "data/amp-pd-genomics/BR-DSNWGS/pgen/br_dsnwgs_hg38.psam",
    "AMP-AD_WGS_Harm": REPO / "data/amp-ad-genomics/WGS_Harmonization/pgen/wgs_harm_hg38.psam",
    "AMP-AD_DivCo_HS": REPO / "data/amp-ad-genomics/DivCo_HS/pgen/divco_hs_hg38.psam",
}

pd.set_option("display.max_rows", 120)
print(f"root exists: {HERE.exists()}   crosswalk: {CROSSWALK.exists()}   grain: {GRAIN.exists()}")
for name, p in CALLSETS.items():
    print(f"  {'ok ' if p.exists() else 'MISSING'} {name}")

## 01 — pull the pseudobulk sample lists

Only needed to refresh. Structure is preserved, never flattened: CMD and rasle put the
cell type in the *directory* and name every file `samples.tsv`/`.csv`, the other four put
it in the *filename*. Flattening would collide and lose the labels step 02 needs.

In [ ]:
# %%bash
# set -euo pipefail
# DEST="$HOME/projects/amp-ad-pd-wgs-gwas/demo_sample_check/pseudobulk/"
# mkdir -p "$DEST"
# rsync -av --prune-empty-dirs \
#   --include='*/' --include='*samples.tsv' --include='*samples.csv' --exclude='*' \
#   ${USER}@biowulf.nih.gov:/data/CARD/sysbio/data/beta1/pseudobulk/ "$DEST"
# echo "files: $(find "$DEST" -name '*samples.[tc]sv' | wc -l | tr -d ' ')  (expected 179)"

## 02 — the 179 sample lists → one donor-level table

Two things make this less trivial than a `cat`:

1. **Six cohorts, six donor columns.** `DONOR_COLS` is tried in order and the first present
   wins — a priority list rather than a per-cohort mapping, so a new cohort needs no edit.
2. **The CMD files are cell-level.** `ID` is a 10x barcode;
   `hypothalamus/c1_4_oligo_mature` holds 144,247 rows for **11 donors**. Taking `ID` would
   inflate the denominator ~1000×. Every cohort is deduplicated to distinct donors per file.

`projid` is carried where present as a second key into the genotypes.

In [ ]:
# First match wins. `sample_id` is last because in the 3 CMD files that lack `donor_id`
# it is the donor, but elsewhere a donor can have several biosamples.
DONOR_COLS = ("donor_id", "participant_id", "individualID", "donor", "sample_id")


def celltype_of(path: Path) -> str:
    """CMD and rasle put the cell type in the directory; the rest in the filename."""
    stem = path.stem
    return path.parent.name if stem == "samples" else stem.removesuffix("_samples")


def norm_projid(s: pd.Series) -> pd.Series:
    """projid is a float in rosmap ('3283241.0') and an int in mitrosmap."""
    n = pd.to_numeric(s, errors="coerce")
    return n.astype("Int64").astype(str).where(n.notna(), "")


def build_donor_table(pseudobulk: Path = PSEUDOBULK) -> pd.DataFrame:
    """One row per (cohort, tissue, celltype, donor). Reads only the columns it needs."""
    files = sorted(pseudobulk.rglob("*samples.[tc]sv"))
    if not files:
        raise FileNotFoundError(f"no sample files under {pseudobulk} — run cell 01 first")

    frames, skipped = [], []
    for f in files:
        rel = f.relative_to(pseudobulk)
        sep = "\t" if f.suffix == ".tsv" else ","
        header = pd.read_csv(f, sep=sep, nrows=0).columns
        col = next((c for c in DONOR_COLS if c in header), None)
        if col is None:
            skipped.append((str(rel), list(header)))
            continue

        use = [col] + (["projid"] if "projid" in header else [])
        df = pd.read_csv(f, sep=sep, usecols=use, dtype=str)
        df = df.rename(columns={col: "donor"})
        df["projid"] = norm_projid(df["projid"]) if "projid" in df else ""
        df["donor"] = df["donor"].str.strip()
        df = df[df["donor"].notna() & (df["donor"] != "")]

        df = df.drop_duplicates(subset=["donor", "projid"])
        df.insert(0, "celltype", celltype_of(f))
        df.insert(0, "tissue", rel.parts[1])
        df.insert(0, "cohort", rel.parts[0].removesuffix("_pseudobulk_out"))
        df["id_column"] = col
        frames.append(df)

    if skipped:
        print("WARNING — no donor column found, files skipped:")
        for rel, cols in skipped:
            print(f"  {rel}  cols={cols}")

    out = pd.concat(frames, ignore_index=True)
    out.to_csv(OUT / "pseudobulk_donors.tsv", sep="\t", index=False)
    (OUT / "pseudobulk_donors_unique.txt").write_text(
        "\n".join(sorted(out["donor"].unique())) + "\n"
    )
    print(f"{len(files)} files -> {len(out)} rows, {out['donor'].nunique()} unique donors")
    return out


donors = build_donor_table()
donors.groupby("cohort").agg(
    rows=("donor", "size"), unique_donors=("donor", "nunique"), groups=("celltype", "nunique")
)

## 03 — one table: every person, every fact

`samples` has **one row per person** and every fact about them as a column, so any question
is a query rather than another bespoke aggregation. Built from four inputs — the callset
psams, `genome_crosswalk.csv`, `analysis_grain.csv`, and the pseudobulk donor list.

The crosswalk is what makes the join possible: a genotype IID is not a person. One
individual is `MAP<8d>` in WGS_Harmonization, `R<7d>` in the pseudobulks, and
`<individualID>_DLPFC_WGS` in DivCo_HS. `clinical_core.py` resolves all of that; this notebook
re-derives none of it.

| column | meaning |
|---|---|
| `<callset name>` | in that psam (6 boolean columns) |
| `in_demo` / `in_current` | in any demo / any current callset |
| `in_grain` | in `analysis_grain.csv` — i.e. **actually analysed** |
| `n_grain_iids` | grain rows for this person; **>1 means duplicate genomes** |
| `ancestry` `sex` `pheno` `dx_detailed` | from the grain |
| `in_pseudobulk` `pb_cohorts` `pb_n_celltypes` `pb_n_tissues` | pseudobulk side |

In [ ]:
def read_ids(path: Path) -> list[str]:
    """IID column of a .psam/.fam, else the first field of a plain list."""
    lines = path.read_text().splitlines()
    head = lines[0].lstrip("#").split("\t")
    if "IID" in head:
        i, body = head.index("IID"), lines[1:]
    elif path.suffix == ".fam":
        i, body = 1, lines
    else:
        i, body = 0, lines
    return [(ln.split("\t") if "\t" in ln else ln.split())[i].strip()
            for ln in body if ln.strip()]


xw = pd.read_csv(CROSSWALK, dtype=str)
# 4 MSBB rows have a null individual_id (their biospecimen lookup failed upstream). Fall
# back to the IID: otherwise all four collapse into a single NaN key and 4 people are
# counted as 1.
xw["individual_id"] = xw["individual_id"].fillna(xw["IID"])
iid2ind = dict(zip(xw["IID"], xw["individual_id"]))

grain = pd.read_csv(GRAIN, dtype=str)
pb = pd.read_csv(OUT / "pseudobulk_donors.tsv", sep="\t", dtype=str)

samples = pd.DataFrame(index=pd.Index(
    sorted(set(iid2ind.values()) | set(grain["individual_id"]) | set(pb["donor"])),
    name="individual_id"))

for name, path in CALLSETS.items():
    samples[name] = samples.index.isin({iid2ind.get(i, i) for i in read_ids(path)})
samples["in_demo"] = samples[[c for c in CALLSETS if c.startswith("demo_")]].any(axis=1)
samples["in_current"] = samples[[c for c in CALLSETS if not c.startswith("demo_")]].any(axis=1)

# n_grain_iids > 1 = the same person under two IIDs (two callsets). 178 of them; they are
# NOT the documented dual-source 87, which are same-IID and already fused to one row.
# `dup_cluster_id` is empty for every row, so nothing downstream removes them, and
# contrasts.csv counts rows — which is why it reads ~180 higher than a person count.
g1 = grain.drop_duplicates(subset="individual_id").set_index("individual_id")
samples["in_grain"] = samples.index.isin(g1.index)
samples["n_grain_iids"] = grain.groupby("individual_id").size().reindex(samples.index).fillna(0).astype(int)
for c in ("ancestry", "sex", "pheno", "dx_detailed", "source_callset"):
    samples[c] = g1[c].reindex(samples.index)

pbg = pb.groupby("donor")
samples["in_pseudobulk"] = samples.index.isin(pb["donor"])
samples["pb_cohorts"] = pbg["cohort"].agg(lambda s: ",".join(sorted(set(s)))).reindex(samples.index)
samples["pb_n_celltypes"] = pbg.size().reindex(samples.index).fillna(0).astype(int)
samples["pb_n_tissues"] = pbg["tissue"].nunique().reindex(samples.index).fillna(0).astype(int)

# ── pre-QC columns, so counts can be reported with OR without this pipeline's QC ──────
# dx_detailed above comes from the grain, which is already post-QC — anyone dropped has no
# phenotype there. individual_core.csv carries clinical data for all 17,160 individuals
# regardless of QC, and qc_outcomes.csv records why each genotype IID was kept or dropped.
#
# NOTE: anc_all is only populated for retained IIDs (12,004 of 13,428) — genotools_callrate
# drops happen BEFORE ancestry assignment. So there is no pre-QC EUR split; a "full"
# ancestry-restricted count does not exist without re-running ancestry.
ic = (pd.read_csv(REPO / "clinical_core_out/individual_core.csv", dtype=str)
        .drop_duplicates("individual_id").set_index("individual_id"))
qc = (pd.read_csv(REPO / "clinical_core_out/qc_outcomes.csv", dtype=str)
        .drop_duplicates("individual_id").set_index("individual_id"))

samples["dx_all"] = ic["dx_detailed"].reindex(samples.index).fillna(samples["dx_detailed"])
samples["anc_all"] = qc["ancestry"].reindex(samples.index).fillna(samples["ancestry"])
samples["qc_status"] = qc["status"].reindex(samples.index)
samples["qc_reason"] = qc["reason"].reindex(samples.index)

samples.to_csv(OUT / "samples.tsv", sep="\t")
print(f"{len(samples)} people x {samples.shape[1]} columns -> out/samples.tsv")
print(f"  in a current callset {int(samples.in_current.sum()):>6}"
      f"   in the GWAS {int(samples.in_grain.sum()):>6}"
      f"   with pseudobulk {int(samples.in_pseudobulk.sum()):>5}")
samples.head()

### Querying it

Everything asked so far, as one-liners against `samples`.

In [ ]:
# how many pseudobulk donors are in the GWAS?
print("pseudobulk in GWAS      :", (samples.in_pseudobulk & samples.in_grain).sum())

# coverage by pseudobulk cohort
display(
    samples[samples.in_pseudobulk]
    .groupby("pb_cohorts")
    .agg(donors=("in_pseudobulk", "size"),
         matched=("in_current", "sum"),
         in_GWAS=("in_grain", "sum"),
         demo=("in_demo", "sum"))
)

# "total N analysed": EUR, AD/PD, split by pseudobulk availability
eur = samples[samples.in_grain & (samples.ancestry == "EUR") & samples.dx_detailed.isin(["AD", "PD"])]
display(
    eur.groupby(["dx_detailed", "in_pseudobulk"]).size().unstack(fill_value=0)
       .rename(columns={False: "no_pseudobulk", True: "with_pseudobulk"})
       .assign(TOTAL=lambda d: d.sum(axis=1))
)

# genotyped but no pseudobulk, per callset
display(
    pd.DataFrame({c: {"people": int(samples[c].sum()),
                      "no_pseudobulk": int((samples[c] & ~samples.in_pseudobulk).sum())}
                  for c in CALLSETS}).T
)

# duplicate genomes entering the GWAS as two samples
print("\npeople with >1 grain IID:", (samples.n_grain_iids > 1).sum(),
      "-> extra rows:", int(samples.n_grain_iids.sum() - samples.in_grain.sum()))

### AD / PD / control counts — demo vs current, with and without pseudobulk

The same query twice, with one filter swapped:

- **FULL** — everyone in the psams, no QC applied. Phenotype from `individual_core.csv`.
- **ANALYSED** — restricted to `in_grain`, i.e. what actually reaches the GWAS.

`demo_TOTAL` is the demo's **pre-QC pool**, not its published N: the demo then applied its
own callrate/related/het chain and a relatedness `--remove`, neither of which is pulled
here. Its true N sits between the two tables.

In [ ]:
DX = ["AD", "PD", "control"]


def dx_table(base, dx_col="dx_all", ancestry=None):
    """AD/PD/control x (with|without pseudobulk), for demo and current side by side."""
    if ancestry is not None:
        base = base & (samples["anc_all"] == ancestry)

    def arm(mask, label):
        d = samples[mask & samples[dx_col].isin(DX)]
        t = (d.groupby([dx_col, "in_pseudobulk"]).size().unstack(fill_value=0)
               .reindex(index=DX, columns=[True, False], fill_value=0))
        t.columns = [f"{label}_with_pb", f"{label}_no_pb"]
        t[f"{label}_TOTAL"] = t.sum(axis=1)
        return t

    out = arm(base & samples.in_demo, "demo").join(arm(base & samples.in_current, "current"))
    out.loc["TOTAL"] = out.sum()
    return out


everyone = pd.Series(True, index=samples.index)
print("FULL — everyone in the psams, no QC filter")
display(dx_table(everyone, "dx_all"))

print("ANALYSED — in analysis_grain (post-QC)")
display(dx_table(samples.in_grain, "dx_detailed"))

print("ANALYSED, EUR only")
display(dx_table(samples.in_grain & (samples.ancestry == "EUR"), "dx_detailed"))

print("QC drops among people in a current psam:")
print(samples[samples.in_current & samples.qc_status.eq("dropped")]
      .qc_reason.value_counts().to_string())

## Coverage by cohort

Bars are donors reached, against the light rule showing every donor in that cohort. The two
all-zero callsets are dropped — `demo_AMP_PD_EUR` because the demo's PD arm is WB-DWGS (the
*living* cohorts) while the only AMP-PD pseudobulk is postmortem, and `AMP-PD_WB-DWGS` for
the same reason.

The gap between bar and rule is unreachable donors. For CMD and rasle it is the whole width,
by design: those are external reference atlases (KPMP, Tadross) and AMP **RA/SLE** — a
different consortium — with no genotypes anywhere in this project.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SERIES = ["demo_AMP_AD_EUR", "AMP-PD_BR-DSNWGS", "AMP-AD_WGS_Harm", "AMP-AD_DivCo_HS"]
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # validated categorical slots 1-4
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d8d7d2"

plot = matrix.drop(index="ALL unique")
cohorts = plot.index.tolist()
y = np.arange(len(cohorts))
h = 0.8 / len(SERIES)

fig, ax = plt.subplots(figsize=(9, 5.5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

for i, c in enumerate(cohorts):  # the "all donors" rule behind each group
    ax.plot([plot.loc[c, "donors"]] * 2, [y[i] - 0.42, y[i] + 0.42], color=RULE, lw=2, zorder=1)

for i, (s, col) in enumerate(zip(SERIES, COLORS)):
    off = 0.4 - h * (i + 0.5)
    vals = plot[s].to_numpy() if s in plot else np.zeros(len(plot))
    ax.barh(y + off, vals, height=h * 0.86, color=col, label=s, zorder=2)
    for yy, v in zip(y + off, vals):  # direct labels — the palette's contrast WARN
        if v:                          # obligates visible values
            ax.text(v + 6, yy, f"{v:.0f}", va="center", fontsize=8, color=MUTED)

for i, c in enumerate(cohorts):
    ax.text(plot.loc[c, "donors"] + 6, y[i] + 0.46, f"{plot.loc[c, 'donors']} donors",
            va="center", fontsize=8, color=MUTED, style="italic")

ax.set_yticks(y, cohorts, fontsize=10, color=INK)
ax.invert_yaxis()
ax.set_xlabel("donors reached", fontsize=9, color=MUTED)
ax.set_xlim(0, plot["donors"].max() * 1.18)
ax.tick_params(colors=MUTED, labelsize=8)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color(RULE)
ax.xaxis.grid(True, color=RULE, lw=0.6)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="lower right", labelcolor=MUTED)
ax.set_title("Pseudobulk donors reached, by cohort and callset", fontsize=11,
             color=INK, loc="left", pad=12)
fig.tight_layout()
plt.show()